In [1]:
import subprocess
from pathlib import Path
root=Path('/tf/wjy')
log=root/'hbm_int8_rerun_20260916.log'
with log.open('a') as out:
    proc=subprocess.Popen(['/tf/wjy/.venv-quant/bin/python','-u',str(root/'hbm_int8_rerun_20260916.py')],cwd=str(root),stdout=out,stderr=subprocess.STDOUT,start_new_session=True)
print('Campaign PID:',proc.pid,'Log:',log)

Campaign PID: 70581 Log: /tf/wjy/hbm_fp16_rerun_20260916.log


In [31]:
from pathlib import Path
lines=Path('/tf/wjy/hbm_int8_rerun_20260916.log').read_text().splitlines()
print('\n'.join(x for x in lines if x.startswith(('START ', 'WORKER ', 'ROUND_STATUS', 'PROGRESS', 'RESULT_JSON', 'EVAL START', 'Traceback', 'RuntimeError', 'WAITING', 'GPU SELECTED')))[-12000:])
print('LAST LOG LINE:', lines[-1] if lines else '(empty)')

aceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
PROGRESS 24 / 164 correct= 0
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
PROGRESS 28 / 164 correct= 0
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
PROGRESS 32 / 164 correct= 1
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Trace

In [35]:
import json
from pathlib import Path

folder = Path("/tf/wjy/results_hbm_int8_rerun_20260916")

for path in sorted(folder.glob("MathQA_*.json")):
    with path.open("r", encoding="utf-8") as f:
        d = json.load(f)

    print(
        path.name,
        "| seed =", d.get("seed"),
        "| round =", d.get("round"),
        "| correct =", d.get("correct"),
        "| accuracy =", f"{d.get('accuracy', 0)*100:.2f}%",
        "| actual_ber =", d.get("actual_ber")
    )

MathQA_2026091201.json | seed = 2026091201 | round = 1 | correct = 116 | accuracy = 11.60% | actual_ber = 0.0030000000081669828
MathQA_2026091202.json | seed = 2026091202 | round = 2 | correct = 44 | accuracy = 4.40% | actual_ber = 0.0030000000081669828
MathQA_2026091203.json | seed = 2026091203 | round = 3 | correct = 28 | accuracy = 2.80% | actual_ber = 0.0030000000081669828
MathQA_2026091204.json | seed = 2026091204 | round = 4 | correct = 7 | accuracy = 0.70% | actual_ber = 0.0030000000081669828
MathQA_2026091205.json | seed = 2026091205 | round = 5 | correct = 95 | accuracy = 9.50% | actual_ber = 0.0030000000081669828


In [36]:
from pathlib import Path
import shutil

src = Path(
    "/tf/wjy/results_hbm_int8_rerun_20260916/"
    "MathQA_2026091204_worker.py"
)

dst = Path(
    "/tf/wjy/MathQA_invalid_analysis_2026091204.py"
)

assert src.exists(), f"找不到原始 worker：{src}"

shutil.copy2(src, dst)

print("原始文件：", src)
print("分析副本：", dst)
print("复制成功：", dst.exists())

原始文件： /tf/wjy/results_hbm_fp16_rerun_20260916/MathQA_2026091204_worker.py
分析副本： /tf/wjy/MathQA_invalid_analysis_2026091204.py
复制成功： True


In [37]:
from pathlib import Path

path = Path("/tf/wjy/MathQA_invalid_analysis_2026091204.py")

code = path.read_text(encoding="utf-8")

old = """    correct = 0
    processed = 0
    for start in range(0, len(test), 4):
        rows = [test[i] for i in range(start, min(start + 4, len(test)))]
        generated = generate_batch([messages(r) for r in rows], 1024)
        correct += sum((parse_answer(text) == row['correct'].strip().lower() for text, row in zip(generated, rows)))
        processed += len(rows)
        print('PROGRESS', start + len(rows), '/', len(test), 'correct=', correct, flush=True)
    return (correct, len(test))"""

new = """    correct = 0
    invalid = 0
    processed = 0

    pred_counts = {
        'a': 0,
        'b': 0,
        'c': 0,
        'd': 0,
        'e': 0
    }

    invalid_examples = []

    for start in range(0, len(test), 4):
        rows = [test[i] for i in range(start, min(start + 4, len(test)))]
        generated = generate_batch([messages(r) for r in rows], 1024)

        for offset, (text, row) in enumerate(zip(generated, rows)):
            pred = parse_answer(text)
            gold = row['correct'].strip().lower()

            if pred is None:
                invalid += 1

                if len(invalid_examples) < 20:
                    invalid_examples.append({
                        'index': start + offset,
                        'gold': gold,
                        'output': repr(text[:1500])
                    })
            else:
                pred_counts[pred] += 1

            if pred == gold:
                correct += 1

        processed += len(rows)

        print(
            'PROGRESS',
            processed,
            '/',
            len(test),
            'correct=',
            correct,
            'invalid=',
            invalid,
            flush=True
        )

    wrong_valid = processed - correct - invalid
    valid = processed - invalid

    print('\\n' + '=' * 70)
    print('MATHQA OUTPUT ANALYSIS')
    print('=' * 70)

    print('Total:', processed)
    print('Correct:', correct)
    print('Wrong valid:', wrong_valid)
    print('Invalid:', invalid)

    print(f'Accuracy: {correct / processed * 100:.2f}%')
    print(f'Valid rate: {valid / processed * 100:.2f}%')
    print(f'Invalid rate: {invalid / processed * 100:.2f}%')

    print('\\nPrediction distribution:')

    for letter in ['a', 'b', 'c', 'd', 'e']:
        count = pred_counts[letter]
        print(
            f'{letter.upper()}: {count} '
            f'({count / processed * 100:.2f}%)'
        )

    print(
        f'Invalid: {invalid} '
        f'({invalid / processed * 100:.2f}%)'
    )

    print('\\n' + '=' * 70)
    print('FIRST 20 INVALID OUTPUTS')
    print('=' * 70)

    if not invalid_examples:
        print('No invalid outputs.')
    else:
        for item in invalid_examples:
            print('\\n' + '-' * 70)
            print('Index:', item['index'])
            print('Gold:', item['gold'])
            print('Raw output:')
            print(item['output'])

    return (correct, len(test))"""

if old not in code:
    raise RuntimeError(
        "没有找到原始 run_evaluation 代码块，停止修改。"
        "不要继续运行分析脚本。"
    )

code = code.replace(old, new, 1)

path.write_text(code, encoding="utf-8")

print("修改成功:", path)
print("已加入：")
print("  1. Invalid 数量/比例")
print("  2. Wrong-valid 数量")
print("  3. A/B/C/D/E 分布")
print("  4. 前20个 Invalid 原始输出")

修改成功: /tf/wjy/MathQA_invalid_analysis_2026091204.py
已加入：
  1. Invalid 数量/比例
  2. Wrong-valid 数量
  3. A/B/C/D/E 分布
  4. 前20个 Invalid 原始输出


In [1]:
import subprocess
import os
from pathlib import Path

PYTHON = "/tf/wjy/.venv-quant/bin/python"
SCRIPT = "/tf/wjy/MathQA_invalid_analysis_2026091204.py"
LOG = "/tf/wjy/MathQA_invalid_analysis_2026091204.log"
SEED = "2026091204"

assert Path(PYTHON).exists()
assert Path(SCRIPT).exists()

# 继承当前环境，并明确指定 GPU 0
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open(LOG, "w", encoding="utf-8")

proc = subprocess.Popen(
    [
        PYTHON,
        "-u",
        SCRIPT,
        SEED
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    cwd="/tf/wjy",
    env=env
)

print("任务重新启动成功")
print("PID:", proc.pid)
print("Seed:", SEED)
print("CUDA_VISIBLE_DEVICES:", env["CUDA_VISIBLE_DEVICES"])
print("日志:", LOG)

任务重新启动成功
PID: 117730
Seed: 2026091204
CUDA_VISIBLE_DEVICES: 0
日志: /tf/wjy/MathQA_invalid_analysis_2026091204.log


In [12]:
from pathlib import Path

log = Path("/tf/wjy/MathQA_invalid_analysis_2026091204.log")

lines = log.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

print("===== 最新40行 =====")

for line in lines[-40:]:
    print(line)

===== 最新40行 =====
Gold: c
Raw output:
'"the teacher\'s age is 1, 5 + 5 = 1, 5 + 5 = 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5

In [13]:
from pathlib import Path

log = Path("/tf/wjy/MathQA_invalid_analysis_2026091204.log")

lines = log.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

# 找到最终分析统计的位置
for i, line in enumerate(lines):
    if "MATHQA OUTPUT ANALYSIS" in line:
        print("\n".join(lines[i:i+20]))

MATHQA OUTPUT ANALYSIS
Total: 1000
Correct: 6
Wrong valid: 36
Invalid: 958
Accuracy: 0.60%
Valid rate: 4.20%
Invalid rate: 95.80%

Prediction distribution:
A: 7 (0.70%)
B: 0 (0.00%)
C: 0 (0.00%)
D: 6 (0.60%)
E: 29 (2.90%)
Invalid: 958 (95.80%)

FIRST 20 INVALID OUTPUTS


In [14]:
from pathlib import Path
import shutil

src = Path(
    "/tf/wjy/results_hbm_int8_rerun_20260916/"
    "MathQA_2026091201_worker.py"
)

dst = Path(
    "/tf/wjy/MathQA_invalid_analysis_2026091201.py"
)

assert src.exists(), f"找不到原始 worker：{src}"

shutil.copy2(src, dst)

print("原始文件：", src)
print("分析副本：", dst)
print("复制成功：", dst.exists())

原始文件： /tf/wjy/results_hbm_fp16_rerun_20260916/MathQA_2026091201_worker.py
分析副本： /tf/wjy/MathQA_invalid_analysis_2026091201.py
复制成功： True


In [15]:
from pathlib import Path

path = Path("/tf/wjy/MathQA_invalid_analysis_2026091201.py")

code = path.read_text(encoding="utf-8")

old = """    correct = 0
    processed = 0
    for start in range(0, len(test), 4):
        rows = [test[i] for i in range(start, min(start + 4, len(test)))]
        generated = generate_batch([messages(r) for r in rows], 1024)
        correct += sum((parse_answer(text) == row['correct'].strip().lower() for text, row in zip(generated, rows)))
        processed += len(rows)
        print('PROGRESS', start + len(rows), '/', len(test), 'correct=', correct, flush=True)
    return (correct, len(test))"""

new = """    correct = 0
    invalid = 0
    processed = 0

    pred_counts = {
        'a': 0,
        'b': 0,
        'c': 0,
        'd': 0,
        'e': 0
    }

    invalid_examples = []

    for start in range(0, len(test), 4):
        rows = [test[i] for i in range(start, min(start + 4, len(test)))]
        generated = generate_batch([messages(r) for r in rows], 1024)

        for offset, (text, row) in enumerate(zip(generated, rows)):
            pred = parse_answer(text)
            gold = row['correct'].strip().lower()

            if pred is None:
                invalid += 1

                if len(invalid_examples) < 20:
                    invalid_examples.append({
                        'index': start + offset,
                        'gold': gold,
                        'output': repr(text[:1500])
                    })
            else:
                pred_counts[pred] += 1

            if pred == gold:
                correct += 1

        processed += len(rows)

        print(
            'PROGRESS',
            processed,
            '/',
            len(test),
            'correct=',
            correct,
            'invalid=',
            invalid,
            flush=True
        )

    wrong_valid = processed - correct - invalid
    valid = processed - invalid

    print('\\n' + '=' * 70)
    print('MATHQA OUTPUT ANALYSIS')
    print('=' * 70)

    print('Total:', processed)
    print('Correct:', correct)
    print('Wrong valid:', wrong_valid)
    print('Invalid:', invalid)

    print(f'Accuracy: {correct / processed * 100:.2f}%')
    print(f'Valid rate: {valid / processed * 100:.2f}%')
    print(f'Invalid rate: {invalid / processed * 100:.2f}%')

    print('\\nPrediction distribution:')

    for letter in ['a', 'b', 'c', 'd', 'e']:
        count = pred_counts[letter]
        print(
            f'{letter.upper()}: {count} '
            f'({count / processed * 100:.2f}%)'
        )

    print(
        f'Invalid: {invalid} '
        f'({invalid / processed * 100:.2f}%)'
    )

    print('\\n' + '=' * 70)
    print('FIRST 20 INVALID OUTPUTS')
    print('=' * 70)

    if not invalid_examples:
        print('No invalid outputs.')
    else:
        for item in invalid_examples:
            print('\\n' + '-' * 70)
            print('Index:', item['index'])
            print('Gold:', item['gold'])
            print('Raw output:')
            print(item['output'])

    return (correct, len(test))"""

if old not in code:
    raise RuntimeError(
        "没有找到原始 run_evaluation 代码块，停止修改。"
    )

code = code.replace(old, new, 1)
path.write_text(code, encoding="utf-8")

print("修改成功:", path)
print("已加入 Invalid / Wrong-valid / A-E分布 / Invalid原始输出统计")

修改成功: /tf/wjy/MathQA_invalid_analysis_2026091201.py
已加入 Invalid / Wrong-valid / A-E分布 / Invalid原始输出统计


In [16]:
import subprocess
import os
from pathlib import Path

python_bin = "/tf/wjy/.venv-quant/bin/python"
script = "/tf/wjy/MathQA_invalid_analysis_2026091201.py"
log_path = "/tf/wjy/MathQA_invalid_analysis_2026091201.log"

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open(log_path, "w", buffering=1)

proc = subprocess.Popen(
    [python_bin, script, "2026091201"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
)

print("已启动")
print("PID:", proc.pid)
print("Seed: 2026091201")
print("Log:", log_path)

已启动
PID: 127513
Seed: 2026091201
Log: /tf/wjy/MathQA_invalid_analysis_2026091201.log
